In [1466]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import t

from src.experiment_runner.models import ExperimentRun, RequestRecord

# Loading functions

In [1467]:
def load_run(path: Path) -> ExperimentRun:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    reqs = [RequestRecord(**r) for r in data["requests"]]
    data["requests"] = reqs

    return ExperimentRun(**data)

In [1468]:
def compute_wa(run: ExperimentRun) -> float:
    replicas = [
        len(r.upstream.get("sockets", [])) or 1
        for r in run.requests
    ]
    return float(np.mean(replicas))

In [1469]:
def extract_row(run: ExperimentRun, group: str) -> dict:
    balancer = run.factors.get("balancer")
    replication = run.factors.get("replication") or "none"
    seed = run.factors.get("seed", -1)

    latency = run.summary["overall"]["latency_ms"]

    p99 = latency["p99"]
    p95 = latency["p95"]
    wa = compute_wa(run)

    return {
        "run_id": run.run_id,
        "seed": seed,
        "algorithm": balancer,
        "strategy": replication,
        "group": group,
        "p99": p99,
        "p95": p95,
        "wa": wa,
    }

In [1470]:
def collect_rows(root: Path) -> pd.DataFrame:
    rows = []

    for file in root.rglob("*.json"):
        if "baseline" in file.parts:
            group = "baseline"
        elif "non_adaptive" in file.parts:
            group = "non_adaptive"
        elif "adaptive" in file.parts:
            group = "adaptive"
        else:
            continue

        run = load_run(file)
        row = extract_row(run, group)

        # batch_id = timestamp папки эксперимента
        row["batch_id"] = file.parents[1].name
        row["file_name"] = file.name

        rows.append(row)

    return pd.DataFrame(rows)

In [1471]:
def aggregate(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df
        .groupby(["algorithm", "strategy", "group"], as_index=False)
        .agg({
            "p99": "mean",
            "wa": "mean",
        })
    )

In [1472]:
def compute_scores(df: pd.DataFrame) -> pd.DataFrame:
    results = []

    for group in ["non_adaptive", "adaptive"]:
        repl = df[df.group == group].copy()
        base = df[df.group == "baseline"].copy()

        if repl.empty:
            continue

        # Парное сравнение по algorithm + seed
        merged = repl.merge(
            base,
            on=["algorithm", "seed"],
            suffixes=("", "_baseline"),
        )

        if merged.empty:
            continue

        merged["p99_improvement_percent"] = (
                (merged["p99_baseline"] - merged["p99"]) / merged["p99_baseline"] * 100
        )

        merged["p95_improvement_percent"] = (
                (merged["p95_baseline"] - merged["p95"]) / merged["p95_baseline"] * 100
        )

        merged["wa_ratio"] = merged["wa"] / merged["wa_baseline"]

        merged["efficiency"] = (
                merged["p99_improvement_percent"] / merged["wa_ratio"]
        )

        merged["group"] = group

        agg = (
            merged
            .groupby(["algorithm", "strategy", "group"], as_index=False)
            .agg(
                p99_improvement_percent=("p99_improvement_percent", "mean"),
                p99_median=("p99_improvement_percent", "median"),
                p99_min=("p99_improvement_percent", "min"),
                p99_max=("p99_improvement_percent", "max"),
                p99_std=("p99_improvement_percent", "std"),

                p95_improvement_percent=("p95_improvement_percent", "mean"),
                p95_median=("p95_improvement_percent", "median"),
                p95_min=("p95_improvement_percent", "min"),
                p95_max=("p95_improvement_percent", "max"),
                p95_std=("p95_improvement_percent", "std"),

                wa=("wa_ratio", "mean"),
                efficiency=("efficiency", "mean"),
                experiments_count=("seed", "count"),
                positive_p99_rate=("p99_improvement_percent", lambda s: float((s > 0).mean())),
                positive_p95_rate=("p95_improvement_percent", lambda s: float((s > 0).mean())),
            )
        )

        n = agg["experiments_count"]
        t_crit_95 = t.ppf(0.975, df=n-1)
        t_crit_99 = t.ppf(0.995, df=n-1)

        ci_half_p99_95 = t_crit_95 * agg["p99_std"].fillna(0.0) / np.sqrt(n)
        low_p99 = agg["p99_improvement_percent"] - ci_half_p99_95
        high_p99 = agg["p99_improvement_percent"] + ci_half_p99_95
        agg["p99_ci95_range"] = (
                low_p99.round(2).astype(str) + " – " + high_p99.round(2).astype(str)
        )

        ci_half_p95_95 = t_crit_95 * agg["p95_std"].fillna(0.0) / np.sqrt(n)
        low_p95 = agg["p95_improvement_percent"] - ci_half_p95_95
        high_p95 = agg["p95_improvement_percent"] + ci_half_p95_95
        agg["p95_ci95_range"] = (
                low_p95.round(2).astype(str) + " – " + high_p95.round(2).astype(str)
        )

        agg["score"] = (
            agg["p99_improvement_percent"] / agg["wa"]
        )

        results.append(agg)

    if not results:
        return pd.DataFrame()

    final = pd.concat(results, ignore_index=True)
    final = final.sort_values(
        by=["algorithm", "group", "p99_improvement_percent"],
        ascending=[True, True, False],
    )
    return final

In [1473]:
def build_paired_comparison(df: pd.DataFrame, group: str = "non_adaptive") -> pd.DataFrame:
    repl = df[df.group == group].copy()
    base = df[df.group == "baseline"].copy()

    merged = repl.merge(
        base,
        on=["algorithm", "seed"],
        suffixes=("", "_baseline"),
    )

    if merged.empty:
        return pd.DataFrame()

    merged["delta_p99"] = merged["p99_baseline"] - merged["p99"]
    merged["delta_p95"] = merged["p95_baseline"] - merged["p95"]

    merged["p99_improvement_percent"] = (
            merged["delta_p99"] / merged["p99_baseline"] * 100
    )
    merged["p95_improvement_percent"] = (
            merged["delta_p95"] / merged["p95_baseline"] * 100
    )

    cols = [
        "algorithm",
        "strategy",
        "seed",
        "p99_baseline",
        "p99",
        "delta_p99",
        "p99_improvement_percent",
        "p95_baseline",
        "p95",
        "delta_p95",
        "p95_improvement_percent",
        "wa_baseline",
        "wa",
    ]
    return merged[cols].sort_values(["algorithm", "strategy", "seed"])

In [1474]:
def analyze_all(root: Path):
    df = collect_rows(root)
    paired = build_paired_comparison(df, group="non_adaptive")

    if df.empty:
        raise ValueError("Нет данных")

    return compute_scores(df), paired

# Results

In [1475]:
# rows = []
#
# for i in (4, 6, 7, 8, 9):
#     GROUP_DIR = Path(f"../assets/experiments/error_new{i}")
#     res, paired = analyze_all(GROUP_DIR)
#
#     rows.append({
#         "group": i,
#         "p99_improvement_percent": res['p99_improvement_percent'].iloc[0],
#         "p99_min": res['p99_min'].iloc[0],
#         "p99_max": res['p99_max'].iloc[0],
#         "p99_ci95_range": res['p99_ci95_range'].iloc[0],
#         "wa": res['wa'].iloc[0],
#         "score": res['score'].iloc[0],
#     })
#
# df = pd.DataFrame(rows)
#
# print(df)
# df

In [1476]:
GROUP_DIR = Path(f"../assets/experiments/error_new13")
res, paired = analyze_all(GROUP_DIR)

res

,algorithm,strategy,group,p99_improvement_percent,p99_median,p99_min,p99_max,p99_std,p95_improvement_percent,p95_median,...,p95_max,p95_std,wa,efficiency,experiments_count,positive_p99_rate,positive_p95_rate,p99_ci95_range,p95_ci95_range,score
1,topsis,hedged,adaptive,17.966364,18.815091,9.545896,24.883208,5.729538,-0.774368,-1.597528,...,4.572912,3.334305,1.50942,11.874219,10,1.0,0.3,13.87 – 22.07,-3.16 – 1.61,11.902826
0,topsis,hedged,non_adaptive,17.112342,17.940249,11.685417,21.586188,3.493397,-3.285694,-4.052241,...,1.746486,2.703228,1.67938,10.184072,10,1.0,0.1,14.61 – 19.61,-5.22 – -1.35,10.189678


In [1477]:
print(paired)
paired

  algorithm strategy   seed  p99_baseline           p99    delta_p99  \
0    topsis   hedged  10000  18660.434997  16479.885384  2180.549613   
1    topsis   hedged  10001  20234.790208  16051.497656  4183.292552   
2    topsis   hedged  10002  17932.347788  14445.927062  3486.420726   
3    topsis   hedged  10003  18780.393217  15059.170242  3721.222975   
4    topsis   hedged  10004  18851.386365  14782.090689  4069.295676   
5    topsis   hedged  10005  18878.001507  15495.710447  3382.291060   
6    topsis   hedged  10006  18401.956913  15997.834220  2404.122693   
7    topsis   hedged  10007  18372.796445  15470.401254  2902.395191   
8    topsis   hedged  10008  18450.574697  15136.127512  3314.447185   
9    topsis   hedged  10009  18488.447829  16051.792985  2436.654844   

   p99_improvement_percent  p95_baseline           p95   delta_p95  \
0                11.685417  10918.016655  10727.335075  190.681580   
1                20.673763  10531.542745  10865.091465 -333.548720 

,algorithm,strategy,seed,p99_baseline,p99,delta_p99,p99_improvement_percent,p95_baseline,p95,delta_p95,p95_improvement_percent,wa_baseline,wa
0,topsis,hedged,10000,18660.434997,16479.885384,2180.549613,11.685417,10918.016655,10727.335075,190.681580,1.746486,1.0,1.6380
1,topsis,hedged,10001,20234.790208,16051.497656,4183.292552,20.673763,10531.542745,10865.091465,-333.548720,-3.167140,1.0,1.6748
2,topsis,hedged,10002,17932.347788,14445.927062,3486.420726,19.442076,10109.690460,10543.121495,-433.431035,-4.287283,1.0,1.6782
3,topsis,hedged,10003,18780.393217,15059.170242,3721.222975,19.814404,10644.282785,10708.760040,-64.477255,-0.605745,1.0,1.6816
4,topsis,hedged,10004,18851.386365,14782.090689,4069.295676,21.586188,10187.508295,10576.385805,-388.877510,-3.817199,1.0,1.6986
5,topsis,hedged,10005,18878.001507,15495.710447,3382.291060,17.916574,10474.291310,11315.728905,-841.437595,-8.033361,1.0,1.6664
6,topsis,hedged,10006,18401.956913,15997.834220,2404.122693,13.064495,10625.403880,11107.131955,-481.728075,-4.533739,1.0,1.7202
7,topsis,hedged,10007,18372.796445,15470.401254,2902.395191,15.797242,10248.667030,10709.396530,-460.729500,-4.495507,1.0,1.7100
8,topsis,hedged,10008,18450.574697,15136.127512,3314.447185,17.963924,10718.713710,10836.762490,-118.048780,-1.101333,1.0,1.6786
9,topsis,hedged,10009,18488.447829,16051.792985,2436.654844,13.179337,10455.157065,10932.133610,-476.976545,-4.562117,1.0,1.6474


In [1478]:
def collect_request_level(root: Path) -> pd.DataFrame:
    rows = []

    for file in root.rglob("*.json"):
        # определяем группу
        if "baseline" in file.parts:
            group = "baseline"
        elif "non_adaptive" in file.parts:
            group = "non_adaptive"
        elif "adaptive" in file.parts:
            group = "adaptive"
        else:
            continue

        run = load_run(file)

        algorithm = run.factors.get("balancer")
        strategy = run.factors.get("replication") or "none"

        for r in run.requests:
            r: RequestRecord

            rows.append({
                "algorithm": algorithm,
                "strategy": strategy,
                "group": group,
                "ok": r.ok,
                "repl_error": r.upstream['replication_error'],
                "balancer_error": r.upstream['balancer_error'] if 'balancer_error' in r.upstream.keys() else None
            })

    return pd.DataFrame(rows)

In [1479]:
df_req = collect_request_level(GROUP_DIR)

error_stats = (
    df_req
    .groupby(["algorithm", "strategy", "group"])
    .agg(
        total=("ok", "count"),
        success=("ok", "sum"),
    )
)

error_stats["errors"] = error_stats["total"] - error_stats["success"]
error_stats["error_rate"] = error_stats["errors"] / error_stats["total"]
error_stats["error_rate_percent"] = (error_stats["error_rate"] * 100).round(2)

all_errors = pd.concat([
    df_req[["algorithm", "strategy", "group", "repl_error"]]
    .rename(columns={"repl_error": "error"}),
    df_req[["algorithm", "strategy", "group", "balancer_error"]]
    .rename(columns={"balancer_error": "error"})
])

all_errors = all_errors[all_errors["error"].notna()]

error_matrix = (
    all_errors
    .groupby(["algorithm", "strategy", "group", "error"])
    .size()
    .unstack(fill_value=0)
)
error_stats = (
    error_stats
    .join(error_matrix, how="left")
    .fillna(0)
)
print(error_stats)
error_stats

                                 total  success  errors  error_rate  \
algorithm strategy group                                              
topsis    hedged   adaptive      50000    46524    3476     0.06952   
                   non_adaptive  50000    46509    3491     0.06982   
          none     baseline      50000    45905    4095     0.08190   

                                 error_rate_percent  deadline_exceeded  \
algorithm strategy group                                                 
topsis    hedged   adaptive                    6.95              267.0   
                   non_adaptive                6.98              491.0   
          none     baseline                    8.19                0.0   

                                 degraded_error_fallback  no_valid_reply  
algorithm strategy group                                                  
topsis    hedged   adaptive                        241.0           925.0  
                   non_adaptive                 

total  success  errors  error_rate  \
algorithm strategy group                                              
topsis    hedged   adaptive      50000    46524    3476     0.06952   
                   non_adaptive  50000    46509    3491     0.06982   
          none     baseline      50000    45905    4095     0.08190   

                                 error_rate_percent  deadline_exceeded  \
algorithm strategy group                                                 
topsis    hedged   adaptive                    6.95              267.0   
                   non_adaptive                6.98              491.0   
          none     baseline                    8.19                0.0   

                                 degraded_error_fallback  no_valid_reply  
algorithm strategy group                                                  
topsis    hedged   adaptive                        241.0           925.0  
                   non_adaptive                    364.0          1859.0  
          none     baseline                          0.0             0.0